In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

In [ ]:
data = pd.read_csv('./aggregated_metrics.csv')

In [ ]:
with open('./config.json') as f:
    config = json.load(f)

In [ ]:
# Final state of each trial — this is what actually varied across runs
finals = data.groupby('Trial').last()
initials = data.groupby('Trial').first()

outcome_cols = [
    "Population Count", "Cumulative Deaths",
    "Deaths from Aging", "Deaths from Competition",
    "Deaths from Starvation", "Deaths from Exposure",
    "Number of Species",
]
# Auto-discover gene columns from CSV — any "Average X" column except Age
gene_cols = [c for c in finals.columns if c.startswith('Average ') and c != 'Average Age']

outcome_stats = finals[outcome_cols].agg(["mean", "std", "min", "max"]).T
outcome_stats.columns = ["Mean", "Std", "Min", "Max"]

gene_drift = pd.DataFrame({
    "Initial mean": initials[gene_cols].mean(),
    "Final mean":   finals[gene_cols].mean(),
    "Final std":    finals[gene_cols].std(),
})
gene_drift["Change"] = gene_drift["Final mean"] - gene_drift["Initial mean"]

print("=== Outcome statistics (final state per trial) ===")
display(outcome_stats.round(2))
print("=== Gene drift across trials ===")
display(gene_drift.round(2))

In [ ]:
plt.figure(figsize=(12, 6))
timesteps = data['Timestep'].unique()

if config["enable_aging"]:
    average_lifespan = data.groupby('Timestep')['Average Lifespan'].mean()
    std_lifespan = data.groupby('Timestep')['Average Lifespan'].std()
    plt.errorbar(timesteps, average_lifespan, yerr=std_lifespan, label='Average Maximum Lifespan', fmt='-o')

    average_age = data.groupby('Timestep')['Average Age'].mean()
    std_age = data.groupby('Timestep')['Average Age'].std()
    plt.errorbar(timesteps, average_age, yerr=std_age, label='Average Age', fmt='-o')

if config["enable_food"]:
    average_metabolism = data.groupby('Timestep')['Average Metabolism'].mean()
    std_metabolism = data.groupby('Timestep')['Average Metabolism'].std()
    plt.errorbar(timesteps, average_metabolism, yerr=std_metabolism, label='Average Metabolism', fmt='-o')

    average_reproduction_threshold = data.groupby('Timestep')['Average Reproduction Threshold'].mean()
    std_reproduction_threshold = data.groupby('Timestep')['Average Reproduction Threshold'].std()
    plt.errorbar(timesteps, average_reproduction_threshold, yerr=std_reproduction_threshold, label='Average Reproduction Threshold', fmt='-o')

if config["enable_movement"]:
    average_speed = data.groupby('Timestep')['Average Speed'].mean()
    std_speed = data.groupby('Timestep')['Average Speed'].std()
    plt.errorbar(timesteps, average_speed, yerr=std_speed, label='Average Speed', fmt='-o')

if config["enable_space_competition"]:
    average_strength = data.groupby('Timestep')['Average Strength'].mean()
    std_strength = data.groupby('Timestep')['Average Strength'].std()
    plt.errorbar(timesteps, average_strength, yerr=std_strength, label='Average Strength', fmt='-o')

average_hardiness = data.groupby('Timestep')['Average Hardiness'].mean()
std_hardiness = data.groupby('Timestep')['Average Hardiness'].std()
plt.errorbar(timesteps, average_hardiness, yerr=std_hardiness, label='Average Hardiness', fmt='-o')

plt.xlabel('Timestep')
plt.ylabel('Value')
plt.title('Time Series of Average Gene Values Across All Trials')
plt.legend()
plt.show()

In [ ]:
# Deaths are cumulative counters â€” mean across trials at each timestep is meaningful, sum is not
plt.figure(figsize=(12, 6))
timesteps = data['Timestep'].unique()

if config["enable_aging"]:
    mean = data.groupby('Timestep')['Deaths from Aging'].mean()
    std  = data.groupby('Timestep')['Deaths from Aging'].std()
    plt.errorbar(timesteps, mean, yerr=std, label='Deaths from Aging', fmt='-o')

if config["enable_space_competition"]:
    mean = data.groupby('Timestep')['Deaths from Competition'].mean()
    std  = data.groupby('Timestep')['Deaths from Competition'].std()
    plt.errorbar(timesteps, mean, yerr=std, label='Deaths from Competition', fmt='-o')

if config["enable_food"]:
    mean = data.groupby('Timestep')['Deaths from Starvation'].mean()
    std  = data.groupby('Timestep')['Deaths from Starvation'].std()
    plt.errorbar(timesteps, mean, yerr=std, label='Deaths from Starvation', fmt='-o')

mean = data.groupby('Timestep')['Deaths from Exposure'].mean()
std  = data.groupby('Timestep')['Deaths from Exposure'].std()
plt.errorbar(timesteps, mean, yerr=std, label='Deaths from Exposure', fmt='-o')

plt.xlabel('Timestep')
plt.ylabel('Mean cumulative deaths (+-std across trials)')
plt.title('Agent Deaths Over Time - Mean Across Trials')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
timesteps = data['Timestep'].unique()
average_species = data.groupby('Timestep')['Number of Species'].mean()
std_species = data.groupby('Timestep')['Number of Species'].std()
plt.errorbar(timesteps, average_species, yerr=std_species, label='Average Number of Species', fmt='-o')
plt.xlabel('Timestep')
plt.ylabel('Number of Species')
plt.title('Number of Species Over Time Across All Trials')
plt.legend()
plt.show()

In [ ]:
# Final species count per trial â€” not a histogram across all timesteps
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

finals['Number of Species'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Species count')
axes[0].set_title('Final Species Count per Trial')
axes[0].tick_params(axis='x', rotation=0)

finals['Population Count'].plot(kind='bar', ax=axes[1], color='seagreen')
axes[1].set_xlabel('Trial')
axes[1].set_ylabel('Population')
axes[1].set_title('Final Population per Trial')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()